In [ ]:
# Install modern modeling and explainability packages
!pip install -q xgboost catboost shap

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Load Dataset from Google Drive (Update file path if needed)
dataset_path = '/content/drive/MyDrive/Students Performance Dataset.csv'
df = pd.read_csv(dataset_path)
print(f"Initial Dataset Shape: {df.shape}")

# 2. Drop Non-Predictive Identifiers
drop_identifiers = ['Student_ID', 'First_Name', 'Last_Name', 'Email']
df = df.drop(columns=[col for col in drop_identifiers if col in df.columns], errors='ignore')

# 3. Create Target (1 = At-Risk [D, F], 0 = Safe [A, B, C])
df['At_Risk'] = df['Grade'].apply(lambda x: 1 if str(x).strip().upper() in ['D', 'F'] else 0)

# 4. Drop Outcome Columns to Prevent Data Leakage
df = df.drop(columns=['Grade', 'Final_Score', 'Total_Score'], errors='ignore')

# 5. Advanced Domain Feature Engineering
# Academic aggregate & consistency
academic_cols = [col for col in ['Assignments_Avg', 'Quizzes_Avg', 'Participation_Score', 'Projects_Score'] if col in df.columns]
if academic_cols:
    df['Academic_Avg'] = df[academic_cols].mean(axis=1)
    df['Academic_Std'] = df[academic_cols].std(axis=1).fillna(0)

# Exam gap (exam panic vs consistent coursework)
if 'Midterm_Score' in df.columns and 'Academic_Avg' in df.columns:
    df['Midterm_to_Coursework_Gap'] = df['Midterm_Score'] - df['Academic_Avg']

# Lifestyle & Effort composite indicators
if 'Study_Hours_per_Week' in df.columns and 'Sleep_Hours_per_Night' in df.columns:
    df['Study_to_Sleep_Ratio'] = df['Study_Hours_per_Week'] / (df['Sleep_Hours_per_Night'] * 7 + 1e-5)

if 'Stress_Level (1-10)' in df.columns and 'Sleep_Hours_per_Night' in df.columns:
    df['Burnout_Risk'] = df['Stress_Level (1-10)'] / (df['Sleep_Hours_per_Night'] + 1e-5)

if 'Attendance (%)' in df.columns:
    df['Low_Attendance_Flag'] = (df['Attendance (%)'] < 75).astype(int)

print(f"Shape after Feature Engineering: {df.shape}")
print("\nTarget Class Distribution (%):")
print(df['At_Risk'].value_counts(normalize=True) * 100)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

y = df['At_Risk']
X = df.drop(columns=['At_Risk'])

# Identify categorical and numerical columns dynamically
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough'
)

# Process features
X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Compute scale_pos_weight for XGBoost to handle class imbalance
neg_count, pos_count = np.bincount(y_train)
scale_weight = neg_count / pos_count

# Base models
lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42)
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_weight, eval_metric='logloss', random_state=42)
cat = CatBoostClassifier(iterations=400, learning_rate=0.04, depth=5, auto_class_weights='Balanced', verbose=0, random_state=42)

# Soft Voting Ensemble
voting_model = VotingClassifier(
    estimators=[('lr', lr), ('xgb', xgb), ('cat', cat)],
    voting='soft'
)

# Stacking Classifier
stacking_model = StackingClassifier(
    estimators=[('lr', lr), ('rf', rf), ('xgb', xgb), ('cat', cat)],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
    cv=5
)

models = {
    "Tuned Logistic Regression": lr,
    "Tuned Random Forest": rf,
    "Tuned XGBoost": xgb,
    "Tuned CatBoost": cat,
    "Soft Voting Classifier": voting_model,
    "Stacking Classifier": stacking_model
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    })

comparison_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
comparison_df

In [ ]:
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix, roc_curve

# Use the Soft Voting Classifier for final inference
final_model = models["Soft Voting Classifier"]
y_probs = final_model.predict_proba(X_test)[:, 1]

# 1. Find the optimal threshold that maximizes F1-Score
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal Threshold for F1: {best_threshold:.3f}")
print(f"Max Achieved F1-Score: {f1_scores[best_idx]:.4f}\n")

# Predict with optimized threshold
y_pred_opt = (y_probs >= best_threshold).astype(int)
print("Classification Report with Optimal Threshold:")
print(classification_report(y_test, y_pred_opt))

# 2. Confusion Matrix & ROC Curve Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f"Confusion Matrix (Threshold = {best_threshold:.2f})")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

fpr, tpr, _ = roc_curve(y_test, y_probs)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_probs):.3f}")
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title("ROC Curve (Voting Classifier)")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import shap

# Compute SHAP values using the trained XGBoost model
explainer = shap.TreeExplainer(models["Tuned XGBoost"])
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_names)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, precision_recall_curve

# Ensure plot styling
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12

# Safely extract models from the existing `models` dictionary
eval_models = {
    "Tuned Logistic Regression": models.get("Tuned Logistic Regression"),
    "Soft Voting Classifier": models.get("Soft Voting Classifier"),
    "Tuned CatBoost": models.get("Tuned CatBoost"),
    "Tuned XGBoost": models.get("Tuned XGBoost"),
    "Tuned Random Forest": models.get("Tuned Random Forest")
}

# Filter out any None values if names slightly differ
eval_models = {k: v for k, v in eval_models.items() if v is not None}

# -------------------------------------------------------------
# FIG 1: Multi-Model ROC Curves Comparison
# -------------------------------------------------------------
plt.figure(figsize=(7, 6), dpi=300)
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728', '#9467bd']

for (name, model), color in zip(eval_models.items(), colors):
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})", color=color, linewidth=1.8)

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.2, label='Random Chance (AUC = 0.500)')
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity / Recall)")
plt.title("ROC Curves across Evaluated Classifiers", fontweight='bold')
plt.legend(loc='lower right', frameon=True)
plt.grid(alpha=0.3, linestyle=':')
plt.tight_layout()
plt.savefig("Fig2_ROC_Curves_Comparison.png", dpi=300)
plt.show()

# -------------------------------------------------------------
# FIG 2: Precision-Recall & F1 vs. Threshold Curve
# -------------------------------------------------------------
voting_model_eval = models.get("Soft Voting Classifier", list(eval_models.values())[0])
probs_voting = voting_model_eval.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, probs_voting)
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
opt_thresh = thresholds[best_idx]

plt.figure(figsize=(7, 5), dpi=300)
plt.plot(thresholds, precisions[:-1], label='Precision', color='#1f77b4', linestyle='--')
plt.plot(thresholds, recalls[:-1], label='Recall', color='#d62728', linestyle='--')
plt.plot(thresholds, f1_scores[:-1], label='F1-Score', color='#2ca02c', linewidth=2.0)
plt.axvline(x=opt_thresh, color='black', linestyle=':', label=f'Optimal Threshold ({opt_thresh:.3f})')

plt.xlabel("Decision Threshold (τ)")
plt.ylabel("Metric Score")
plt.title("Threshold Optimization Curve (Soft Voting Classifier)", fontweight='bold')
plt.legend(loc='lower left', frameon=True)
plt.grid(alpha=0.3, linestyle=':')
plt.tight_layout()
plt.savefig("Fig3_Threshold_Optimization.png", dpi=300)
plt.show()

# -------------------------------------------------------------
# FIG 3: Optimized Confusion Matrix (τ = opt_thresh)
# -------------------------------------------------------------
y_pred_opt = (probs_voting >= opt_thresh).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)

plt.figure(figsize=(6, 5), dpi=300)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            annot_kws={'size': 14, 'weight': 'bold'},
            xticklabels=['Predicted Safe (0)', 'Predicted At-Risk (1)'],
            yticklabels=['Actual Safe (0)', 'Actual At-Risk (1)'])
plt.ylabel("Ground Truth")
plt.xlabel("Model Classification")
plt.title(f"Confusion Matrix at Optimal Threshold (τ = {opt_thresh:.3f})", fontweight='bold')
plt.tight_layout()
plt.savefig("Fig4_Confusion_Matrix_Optimized.png", dpi=300)
plt.show()

# -------------------------------------------------------------
# FIG 4: Logistic Regression Odds Ratios (Feature Impact)
# -------------------------------------------------------------
lr_model_eval = models.get("Tuned Logistic Regression")

if lr_model_eval is not None and hasattr(lr_model_eval, 'coef_'):
    coefs = lr_model_eval.coef_[0]
    clean_names = [name.replace('num__', '').replace('cat__', '') for name in feature_names]

    df_or = pd.DataFrame({
        'Feature': clean_names,
        'LogOdds': coefs,
        'OR': np.exp(coefs)
    })
    df_or['AbsImpact'] = df_or['LogOdds'].abs()
    df_top = df_or.sort_values(by='AbsImpact', ascending=False).head(10).sort_values(by='OR', ascending=True)

    plt.figure(figsize=(8, 5.5), dpi=300)
    bar_colors = ['#d9534f' if or_val > 1.0 else '#5cb85c' for or_val in df_top['OR']]
    plt.barh(df_top['Feature'], df_top['OR'], color=bar_colors, edgecolor='black', linewidth=0.6)
    plt.axvline(x=1.0, color='black', linestyle='--', linewidth=1.2, label='Neutral Effect (OR = 1.0)')

    plt.xlabel("Odds Ratio (exp(β))")
    plt.ylabel("Key Predictors")
    plt.title("Top Predictors of Academic Risk (Odds Ratios)", fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(axis='x', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.savefig("Fig5_Odds_Ratio_Summary.png", dpi=300)
    plt.show()

In [ ]:

# Assuming raw dataframe is `df` with 'At_Risk' (1 = At-Risk, 0 = Safe)
numeric_cols = [
    'Midterm_Score', 'Assignments_Avg', 'Quizzes_Avg', 'Projects_Score',
    'Attendance (%)', 'Study_Hours_per_Week', 'Stress_Level (1-10)', 'Sleep_Hours_per_Night'
]

summary_stats = []
for col in [c for c in numeric_cols if c in df.columns]:
    total_mean = df[col].mean()
    total_std = df[col].std()

    safe_mean = df[df['At_Risk'] == 0][col].mean()
    safe_std = df[df['At_Risk'] == 0][col].std()

    risk_mean = df[df['At_Risk'] == 1][col].mean()
    risk_std = df[df['At_Risk'] == 1][col].std()

    summary_stats.append({
        'Feature': col,
        'Total Mean ± SD': f"{total_mean:.2f} ± {total_std:.2f}",
        'Safe (Class 0) Mean ± SD': f"{safe_mean:.2f} ± {safe_std:.2f}",
        'At-Risk (Class 1) Mean ± SD': f"{risk_mean:.2f} ± {risk_std:.2f}"
    })

pd.DataFrame(summary_stats)
